In [25]:
import fastf1
import pandas as pd

fastf1.Cache.enable_cache('./cache')

### Functions

In [27]:
def enrich_with_weather_and_telemetry(laps_df, sessions):
    combined_telemetry_list = []

    for session in sessions:
        weather_df = session.weather_data.copy()
        weather_df['Time'] = pd.to_timedelta(weather_df['Time'])  # ensure proper type

        # Match weather per lap
        session_laps = laps_df[
            (laps_df['Year'] == session.date.year) &
            (laps_df['Race'] == session.event['EventName'])
        ].copy()
        session_laps['Time'] = session_laps['LapStartTime'].dt.total_seconds()

        # Convert weather time to seconds to merge
        weather_df['WeatherTime'] = weather_df['Time'].dt.total_seconds()

        # Merge each lap with nearest weather row (on absolute time delta)
        session_with_weather_laps = pd.merge_asof(
            session_laps.sort_values('Time'),
            weather_df.sort_values('WeatherTime'),
            left_on='Time',
            right_on='WeatherTime',
            direction='nearest',
            tolerance=60  # 1 minute
        )

        for index, lap_row in session_with_weather_laps.iterrows():
            try:
                lap = session.laps[
                    (session.laps['Driver'] == lap_row['Driver']) &
                    (session.laps['LapNumber'] == lap_row['LapNumber'])
                ].iloc[0]

                telemetry = lap.get_car_data().add_distance()

                for __, telemetry_row in telemetry.iterrows():

                    telemetry_lap = lap_row.copy()

                    telemetry_lap['Speed'] = telemetry_row['Speed']
                    telemetry_lap['RPM'] = telemetry_row['RPM']
                    telemetry_lap['Throttle'] = telemetry_row['Throttle']
                    telemetry_lap['Brake'] = telemetry_row['Brake']
                    telemetry_lap['nGear'] = telemetry_row['nGear']
                    telemetry_lap['Distance'] = telemetry_row['Distance']
                    telemetry_lap['DRS'] = telemetry_row['DRS']

                    combined_telemetry_list.append(telemetry_lap)

            except (IndexError, AttributeError, KeyError):
                continue  # Skip if data missing        

        print(f"Finished adding weather and telemetry for {session.event['EventName']} {session.date.year}.")

    combined_telemetry_dicts = [s.to_dict() for s in combined_telemetry_list]
    return pd.DataFrame(combined_telemetry_dicts)

### Set variables

In [28]:
races = {'saudi' : 'Saudi Arabian Grand Prix', 'canada' : 'Canadian Grand Prix', 'australia' : 'Australian Grand Prix', 'bahrain' : 'Bahrain Grand Prix', 'spain' : 'Spanish Grand Prix', 'japan' : 'Japanese Grand Prix', 'monaco' : 'Monaco Grand Prix'}

years = [2022, 2023, 2024, 2025]

# Columns to keep
cols_to_keep = ['Driver', 'Team', 'Year', 'Race',
    'Sector1Time', 'Sector2Time', 'Sector3Time',
    'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime',
    'SpeedI1', 'SpeedI2', 'SpeedFL',
    'Stint', 'LapTime', 'LapNumber', 'Position',
    'Compound', 'TyreLife', 'FreshTyre', 'TrackStatus',
    'AirTemp', 'Humidity', 'Pressure', 'Rainfall',
    'TrackTemp', 'WindDirection', 'WindSpeed',
    'Speed', 'RPM', 'Throttle', 'Brake', 'nGear', 'Distance', 'DRS'
]

In [29]:

for race_key, race_name in races.items():
    all_laps = []
    failed_races = []
    sessions = []
    for year in years:
        try:
            session = fastf1.get_session(year, race_name, 'R')
            session.load(laps=True, weather=True, telemetry=True)
            sessions.append(session)
            if hasattr(session, 'laps') and len(session.laps) > 0:
                df = session.laps.copy()
                df['Year'] = year
                df['Race'] = race_name
                all_laps.append(df)
            else:
                failed_races.append((year, race_name))
        except Exception as e:
            failed_races.append((year, race_name))

        if all_laps:
            laps_df = pd.concat(all_laps, ignore_index=True)
            print(f"Total laps loaded: {len(laps_df)}")
        else:
            print("No lap data loaded.")

    # Add Telemetry and Weather Data
    weather_telemetry_enriched_df = enrich_with_weather_and_telemetry(laps_df, sessions)

    # Save to csv for later use
    weather_telemetry_enriched_df.to_csv('data/f1_laps_telemetry_'+race_key+'.csv', index=False)  

    final_df = weather_telemetry_enriched_df

    filtered_df = final_df[cols_to_keep]

    # Group by Year, Race, Driver, Stint and get min TyreLife per group
    filtered_df.groupby(['Year', 'Race', 'Driver', 'Stint'])['TyreLife'].min().sort_values()

    # Grouping by Year, Race, Driver, Stint,
    # LapsTilPit = max(LapNumber in stint) - LapNumber + 1
    labelled_df = filtered_df.copy()

    labelled_df['LapsTilPit'] = (
        filtered_df.groupby(['Year', 'Race', 'Driver', 'Stint'])['LapNumber']
        .transform(lambda x: x.max() - x + 1)
    )

    df_clean = labelled_df.dropna(subset=['LapTime'])

    print(f"Shape before: {labelled_df.shape}")
    print(f"Shape after:  {df_clean.shape}")

    # Save labelled and cleaned data to CSV
    df_clean.to_csv('data/f1_laps_telemetry_'+race_key+'_clean.csv', index=False)
    

core           INFO 	Loading data for Monaco Grand Prix - Race [v3.5.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['11', '55', '1', '16', '63', '4', '14', '44', '77', '5', '10', '31', '3', '18', '6', '24', '22', '23', '47', '20']
core           INFO 	Loading data for Monaco Grand Prix - Race [

Total laps loaded: 1179


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '14', '31', '44', '63', '16', '10', '55', '4', '81', '77', '21', '24', '23', '22', '11', '27', '2', '20', '18']


Total laps loaded: 2694


core           INFO 	Loading data for Monaco Grand Prix - Race [v3.5.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['16', '81', '55', '4', '63', '1', '44', '22', '23', '10', '14', '3', '77', '18', '2', '24', '31', '11', '27', '20']
core           INFO 	Loading data for Monaco Grand Prix - Race 

Total laps loaded: 3931


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '16', '81', '1', '44', '6', '31', '30', '23', '55', '63', '87', '43', '5', '18', '27', '22', '12', '14', '10']


Total laps loaded: 5356
Finished adding weather and telemetry for Monaco Grand Prix 2022.
Finished adding weather and telemetry for Monaco Grand Prix 2023.
Finished adding weather and telemetry for Monaco Grand Prix 2024.
Finished adding weather and telemetry for Monaco Grand Prix 2025.
Shape before: (1954041, 36)
Shape after:  (1817706, 36)
